# Laboratorio 10 — Aprendizaje Semi-Supervisado
**CC3074 Minería de Datos | UVG Semestre I – 2026**

**Dataset:** Dry Bean Dataset — Koklu & Ozkan (2020)  
**Fuente:** https://archive.ics.uci.edu/dataset/602/dry+bean+dataset  
**Persona 1:** Dataset, EDA, Preprocesamiento y Modelo Supervisado Baseline

## 1. Importacion inicial de datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')

df = pd.read_excel('DryBeanDataset/Dry_Bean_Dataset.xlsx')

print('Shape:', df.shape)
print('\nColumnas:', df.columns.tolist())
df.head()

ModuleNotFoundError: No module named 'pandas'

In [ ]:
print('Tipos de datos:')
print(df.dtypes)
print('\nValores nulos por columna:')
print(df.isnull().sum())
print('\nDuplicados:', df.duplicated().sum())

## 2. Inspeccion del conjunto de datos

El dataset contiene mediciones morfologicas de 13,611 granos de frijol obtenidas mediante
vision por computadora. Se clasifican en 7 variedades: SEKER, BARBUNYA, BOMBAY, CALI,
HOROZ, SIRA y DERMASON. Las 16 variables son completamente numericas (areas, perimetros,
factores de forma), lo que lo hace ideal para algoritmos semi-supervisados basados en
distancia y grafos.

In [ ]:
# Estadisticas descriptivas
df.describe().round(4)

In [ ]:
# Balance de clases
class_counts = df['Class'].value_counts()
print('Distribucion de clases:')
print(class_counts)
print(f'\nClase mas frecuente: {class_counts.idxmax()} ({class_counts.max()} muestras)')
print(f'Clase menos frecuente: {class_counts.idxmin()} ({class_counts.min()} muestras)')
print(f'Ratio desbalance: {class_counts.max() / class_counts.min():.2f}x')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

class_counts.plot(kind='bar', ax=axes[0], color=sns.color_palette('Set2', 7), edgecolor='black')
axes[0].set_title('Conteo por clase de frijol')
axes[0].set_xlabel('Variedad')
axes[0].set_ylabel('Cantidad')
axes[0].tick_params(axis='x', rotation=30)

class_counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                  colors=sns.color_palette('Set2', 7), startangle=90)
axes[1].set_title('Proporcion de cada variedad')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distribuciones de todas las variables numericas
features = df.drop(columns=['Class'])
n_cols = 4
n_rows = int(np.ceil(len(features.columns) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(features.columns):
    axes[i].hist(df[col], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribucion de variables morfologicas', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Mapa de correlacion entre features
plt.figure(figsize=(14, 11))
corr = features.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, annot_kws={'size': 7})
plt.title('Correlacion entre variables morfologicas', fontsize=13)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Pares con alta correlacion (>0.95)
high_corr = []
for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.95:
            high_corr.append((corr.columns[i], corr.columns[j], round(corr.iloc[i, j], 3)))

print('Pares con correlacion > 0.95:')
for a, b, c in high_corr:
    print(f'  {a} — {b}: {c}')

In [ ]:
# Boxplots de features clave por variedad de frijol
key_features = ['Area', 'Perimeter', 'MajorAxisLength', 'AspectRation', 'Eccentricity', 'roundness']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
palette = sns.color_palette('Set2', 7)

for i, feat in enumerate(key_features):
    sns.boxplot(x='Class', y=feat, data=df, ax=axes[i], palette=palette)
    axes[i].set_title(f'{feat} por variedad')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=30)

plt.suptitle('Distribucion de features por variedad de frijol', fontsize=14)
plt.tight_layout()
plt.savefig('boxplots_by_class.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualizacion PCA 2D para entender la separabilidad entre clases
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X_temp = StandardScaler().fit_transform(features)
pca_2d = PCA(n_components=2)
X_pca = pca_2d.fit_transform(X_temp)

pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
pca_df['Class'] = df['Class'].values

plt.figure(figsize=(11, 7))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='Class',
                palette='Set2', alpha=0.5, s=15)
plt.title(f'PCA 2D — varianza explicada: PC1={pca_2d.explained_variance_ratio_[0]*100:.1f}%, '
          f'PC2={pca_2d.explained_variance_ratio_[1]*100:.1f}%')
plt.legend(loc='upper right', markerscale=2)
plt.tight_layout()
plt.savefig('pca_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Transformacion y acondicionamiento de datos

Las features tienen escalas muy distintas (Area en decenas de miles, ShapeFactors < 0.01).
Se aplica `StandardScaler` para normalizar. No se requiere encoding de categoricas porque
todas las features son numericas. La columna `Class` se codifica con `LabelEncoder`.

In [ ]:
X = df.drop(columns=['Class']).values
y_raw = df['Class'].values

# Codificar target
le = LabelEncoder()
y = le.fit_transform(y_raw)

print('Clases codificadas:')
for i, cls in enumerate(le.classes_):
    print(f'  {i} -> {cls}')

# Normalizar features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'\nMedia antes de escalar (Area): {X[:, 0].mean():.0f}')
print(f'Media despues de escalar (Area): {X_scaled[:, 0].mean():.4f}')
print(f'Std despues de escalar (Area): {X_scaled[:, 0].std():.4f}')

In [ ]:
# Comparar distribuciones antes y despues del escalado (Area como ejemplo)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(X[:, 0], bins=50, color='salmon', edgecolor='white')
axes[0].set_title('Area — antes de escalar')
axes[0].set_xlabel('Valor original')

axes[1].hist(X_scaled[:, 0], bins=50, color='steelblue', edgecolor='white')
axes[1].set_title('Area — despues de StandardScaler')
axes[1].set_xlabel('Valor estandarizado (z-score)')

plt.suptitle('Efecto del StandardScaler', fontsize=12)
plt.tight_layout()
plt.savefig('scaling_effect.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Configuracion del escenario de etiquetado parcial

Para simular el aprendizaje semi-supervisado:
- Se divide el dataset en 80% entrenamiento / 20% prueba.
- Del conjunto de entrenamiento, solo el **10%** tendra etiqueta visible.
- El 90% restante se tratara como no etiquetado (se marca con `-1`,
  convencion requerida por scikit-learn para semi-supervisado).
- Se guardaran los indices de etiquetados para que Persona 2 y Persona 3
  puedan reutilizar exactamente la misma particion.

In [ ]:
# Split train / test (estratificado para mantener balance de clases)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Tamanio train: {X_train.shape[0]} muestras')
print(f'Tamanio test:  {X_test.shape[0]} muestras')

In [ ]:
# Evaluar multiples proporciones de etiquetado
LABELED_RATIO = 0.10   # <-- cambiar a 0.05, 0.10 o 0.20 para experimentar
np.random.seed(42)

n_labeled = int(len(X_train) * LABELED_RATIO)
labeled_idx = np.random.choice(len(X_train), n_labeled, replace=False)
unlabeled_idx = np.setdiff1d(np.arange(len(X_train)), labeled_idx)

# y con -1 en posiciones no etiquetadas (formato sklearn semi-supervisado)
y_train_ssl = np.full(len(X_train), -1, dtype=int)
y_train_ssl[labeled_idx] = y_train[labeled_idx]

print(f'Porcentaje etiquetado: {LABELED_RATIO*100:.0f}%')
print(f'Muestras etiquetadas:   {n_labeled}')
print(f'Muestras no etiquetadas: {len(unlabeled_idx)}')

# Verificar balance de clases en el subconjunto etiquetado
labeled_classes, labeled_counts = np.unique(y_train[labeled_idx], return_counts=True)
print('\nDistribucion en datos etiquetados:')
for cls, cnt in zip(le.classes_[labeled_classes], labeled_counts):
    print(f'  {cls}: {cnt}')

In [ ]:
# Grafico comparativo: datos etiquetados vs no etiquetados en espacio PCA
pca_train = PCA(n_components=2).fit_transform(X_train)

plt.figure(figsize=(11, 6))
plt.scatter(pca_train[unlabeled_idx, 0], pca_train[unlabeled_idx, 1],
            c='lightgray', s=8, alpha=0.4, label='No etiquetado')
plt.scatter(pca_train[labeled_idx, 0], pca_train[labeled_idx, 1],
            c=y_train[labeled_idx], cmap='Set1', s=30, alpha=0.9,
            edgecolors='black', linewidths=0.3, label='Etiquetado')
plt.title(f'Escenario semi-supervisado: {LABELED_RATIO*100:.0f}% etiquetado (PCA 2D)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(markerscale=2)
plt.tight_layout()
plt.savefig('semi_supervised_split_pca.png', dpi=150, bbox_inches='tight')
plt.show()

---
## ⚠️ STOP — COMMIT AQUI

```bash
git add Persona1.ipynb class_distribution.png feature_distributions.png \
        correlation_heatmap.png boxplots_by_class.png pca_visualization.png \
        scaling_effect.png semi_supervised_split_pca.png
git commit -m "Add: dataset loading, EDA, preprocessing and semi-supervised split"
```
---

## 5. Clasificador de referencia supervisado

El modelo baseline se entrena **unicamente con el subconjunto etiquetado** (10% del train),
sin aprovechar los datos no etiquetados. Sirve como punto de comparacion para medir
cuanto mejoran los algoritmos semi-supervisados al explotar los datos sin etiqueta.

In [ ]:
# Entrenar Random Forest solo con las muestras etiquetadas
X_labeled = X_train[labeled_idx]
y_labeled = y_train[labeled_idx]

baseline = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
baseline.fit(X_labeled, y_labeled)

y_pred_baseline = baseline.predict(X_test)

acc = accuracy_score(y_test, y_pred_baseline)
f1  = f1_score(y_test, y_pred_baseline, average='weighted')
rec = recall_score(y_test, y_pred_baseline, average='weighted')

print('=== BASELINE SUPERVISADO (10% datos etiquetados) ===')
print(f'Accuracy : {acc:.4f}')
print(f'F1-score : {f1:.4f}')
print(f'Recall   : {rec:.4f}')
print()
print(classification_report(y_test, y_pred_baseline, target_names=le.classes_))

In [ ]:
# Matriz de confusion del baseline
fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, y_pred_baseline),
    display_labels=le.classes_
).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Matriz de confusion — Baseline supervisado', fontsize=12)
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('confusion_matrix_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Importancia de features en el baseline
feat_importance = pd.Series(
    baseline.feature_importances_,
    index=df.drop(columns=['Class']).columns
).sort_values(ascending=False)

plt.figure(figsize=(11, 5))
sns.barplot(x=feat_importance.values, y=feat_importance.index, palette='viridis')
plt.title('Importancia de features — Random Forest Baseline')
plt.xlabel('Importancia (Gini)')
plt.tight_layout()
plt.savefig('feature_importance_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 features mas importantes:')
print(feat_importance.head())

In [ ]:
# Desempeno del baseline segun % de datos etiquetados
ratios = [0.05, 0.10, 0.15, 0.20]
baseline_accs = []

for r in ratios:
    n = int(len(X_train) * r)
    idx = np.random.RandomState(42).choice(len(X_train), n, replace=False)
    clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    clf.fit(X_train[idx], y_train[idx])
    baseline_accs.append(accuracy_score(y_test, clf.predict(X_test)))
    print(f'{r*100:.0f}% etiquetado -> Accuracy: {baseline_accs[-1]:.4f}')

plt.figure(figsize=(8, 5))
plt.plot([r*100 for r in ratios], baseline_accs, marker='o',
         color='tomato', linewidth=2, markersize=8, label='Baseline (RF supervisado)')
plt.xlabel('% de datos etiquetados')
plt.ylabel('Accuracy')
plt.title('Baseline: Accuracy vs porcentaje de etiquetado')
plt.ylim(0, 1)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig('baseline_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Guardar variables compartidas para Persona 2 y Persona 3
import pickle

shared = {
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test,
    'y_train_ssl': y_train_ssl,
    'labeled_idx': labeled_idx,
    'unlabeled_idx': unlabeled_idx,
    'le': le,
    'y_pred_baseline': y_pred_baseline,
    'baseline_accs': baseline_accs,
    'ratios': ratios,
    'LABELED_RATIO': LABELED_RATIO
}

with open('shared_data.pkl', 'wb') as f:
    pickle.dump(shared, f)

print('Datos guardados en shared_data.pkl')
print('Persona 2 y Persona 3 deben cargar este archivo al inicio de su notebook.')

---
## ⚠️ STOP — COMMIT AQUI

```bash
git add Persona1.ipynb confusion_matrix_baseline.png \
        feature_importance_baseline.png baseline_curve.png shared_data.pkl
git commit -m "Add: supervised baseline model with metrics, confusion matrix and shared data"
```
---